In [ ]:
# Centralized imports (cleaned)
from bioio import BioImage
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tifffile import imwrite, imread
import tifffile
from skimage.segmentation import expand_labels, clear_border
from skimage.measure import regionprops_table
from cellpose import models
import napari
from liffile import LifFile
import matplotlib.colors as mcolors
import colorsys
from scipy.spatial import KDTree
from skimage.measure import regionprops
from skimage.transform import resize, rescale
from pathlib import Path
import pandas as pd
import scipy.ndimage as ndi
from skimage.measure import regionprops
from instanseg import InstanSeg
import time
from cellpose import models, utils as cellpose_utils
from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_3d_segmentation

available = torch.cuda.is_available()
device_count = torch.cuda.device_count() if available else 0
device_name = torch.cuda.get_device_name(0) if available and device_count > 0 else None

status = {
    "cuda_available": available,
    "device_count": device_count,
    "device_name": device_name,
}
print(status)

{'cuda_available': True, 'device_count': 1, 'device_name': 'NVIDIA GeForce RTX 5080'}


In [ ]:

_MODEL_CACHE = {} # cache loaded models across instances so we don't have to reload them for each image

class SegmentationComparisons:
    """Run several nuclear-segmentation methods on the same image and compare them.

    The input image is a 2-channel z-stack stored as (Z, C, Y, X), where
    channel `dapi_channel` is DAPI and channel `bf_channel` is brightfield/phase.
    Every method writes its label mask to `output_dir` and records the object
    count in `self.counts`.
    """

    def __init__(self, input_csv, index, scale_factor_xy=3, scale_factor_z=2,
        custom_model_path=r"C:\Users\taylorhearn\git_repos\image_quantification\New_Spacefish\cellpose_model",
        output_dir="comparison_masks", dapi_channel=0, bf_channel=1):
        self.input_csv = input_csv
        self.index = index
        self.scale_factor_xy = float(scale_factor_xy)
        self.scale_factor_z = float(scale_factor_z)
        self.custom_model_path = custom_model_path

        row = input_csv.loc[index]
        self.two_channel_image_path = Path(row["2_channel_tif_save_path"])
        self.dapi_channel = int(dapi_channel)
        self.bf_channel = int(bf_channel)
        self.stem = self.row["image_name"]

        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # (Z, C, Y, X)
        self.two_channel_image = tifffile.imread(self.two_channel_image_path)
        if self.two_channel_image.ndim != 4:
            raise ValueError(f"Expected a 4D (Z, C, Y, X) image, got shape {self.two_channel_image.shape}")

        self.results = {}  # method name -> label array
        self.counts = {}   # method name -> object count
    

    def pixel_size(self):
        with tifffile.TiffFile(self.two_channel_image_path) as tif:
            tags = {tag.name: tag.value for tag in tif.pages[0].tags.values()}
            x_um = 1 / (tags["XResolution"][0] / tags["XResolution"][1])
            y_um = 1 / (tags["YResolution"][0] / tags["YResolution"][1])
            try:
                z_um = float(str(tags["IJMetadata"]).split("nscales=")[1].split(",")[2].split("\\nunit")[0])
            except Exception:
                z_um = float(str(tags["ImageDescription"]).split("spacing=")[1].split("loop")[0])
        self.original_spacing = (x_um, y_um, z_um)

    def rescale_image(self):
        """Downsample DAPI and brightfield channels and compute z-anisotropy."""
        self.pixel_size()
        x_um, y_um, z_um = self.original_spacing
        xy_ratio = 1.0 / self.scale_factor_xy
        z_ratio = 1.0 / self.scale_factor_z

        dapi = self.two_channel_image[:, self.dapi_channel].astype(np.float32)  # (Z, Y, X)
        bf = self.two_channel_image[:, self.bf_channel].astype(np.float32)

        self.dapi_ds = rescale(dapi, (z_ratio, xy_ratio, xy_ratio), anti_aliasing=True, preserve_range=True).astype(np.float32)
        self.bf_ds = rescale(bf, (z_ratio, xy_ratio, xy_ratio), anti_aliasing=True, preserve_range=True).astype(np.float32)

        # channel-last 2-channel stack for cellpose-SAM: (Z, Y, X, 2) = [DAPI, BF]
        self.two_channel_ds = np.stack([self.dapi_ds, self.bf_ds], axis=-1)

        # physical voxel spacing after downsampling
        self.z_spacing_ds = z_um * self.scale_factor_z
        self.xy_spacing_ds = x_um * self.scale_factor_xy
        self.anisotropy = self.z_spacing_ds / self.xy_spacing_ds
        self.pixel_size_um_2d = float(self.xy_spacing_ds)
        print(f"downsampled DAPI shape {self.dapi_ds.shape}, anisotropy {self.anisotropy:.3f}")



    def _save_and_record(self, name, labels):
        labels = np.asarray(labels, dtype=np.uint32)
        out_path = self.output_dir / f"{self.stem}_{name}.tif"
        imwrite(out_path, labels)
        self.results[name] = labels
        self.counts[name] = int(labels.max())
        print(f"[{name}] {self.counts[name]} objects -> {out_path.name}")
        return labels

    @staticmethod
    def _get_cellpose_model(pretrained=None):
        key = str(pretrained)
        if key not in _MODEL_CACHE:
            if pretrained is None:
                _MODEL_CACHE[key] = models.CellposeModel(gpu=True)  # built-in CPSAM
            else:
                _MODEL_CACHE[key] = models.CellposeModel(gpu=True, pretrained_model=str(pretrained))
        return _MODEL_CACHE[key]

    def cellpose_sam_2d_stitched(self, min_size=500, stitch_threshold=0.1): # cellpose SAM, 2d stiched, no custom model
        model = self._get_cellpose_model()
        masks, _, _ = model.eval(self.two_channel_ds, channel_axis=3, z_axis=0, do_3D=False, stitch_threshold=stitch_threshold, anisotropy=self.anisotropy, min_size=min_size)
        return self._save_and_record("cpsam_2dstitch", masks)

    def cellpose_sam_true_3d(self, min_size=500): # cellpose SAM, true 3d, no custom model
        model = self._get_cellpose_model()
        masks, _, _ = model.eval(self.two_channel_ds, channel_axis=3, z_axis=0, do_3D=True, anisotropy=self.anisotropy, min_size=min_size)
        return self._save_and_record("cpsam_true3d", masks)

    def cellpose_custom_3d(self, min_size=500, batch_size=128, resample=False): # cellpose custom model on DAPI only, extended to true 3d
        model = self._get_cellpose_model(self.custom_model_path)
        masks, _, _ = model.eval(self.dapi_ds, z_axis=0, do_3D=True, anisotropy=self.anisotropy, min_size=min_size, batch_size=int(batch_size), resample=bool(resample))
        return self._save_and_record("cellpose_custom_3d", masks)

    def instanseg_2d_stitched(self, stitch_threshold=0.1): # instanseg 2d stiched, DAPI only, no custom model
        self._ensure_prepared()

        if "instanseg" not in _MODEL_CACHE:
            _MODEL_CACHE["instanseg"] = InstanSeg("fluorescence_nuclei_and_cells", verbosity=0)
        model = _MODEL_CACHE["instanseg"]
        z_masks = []
        for z in range(self.dapi_ds.shape[0]):
            plane = self.dapi_ds[z][None]  # (C=1, H, W), DAPI only
            labeled, _ = model.eval_small_image(plane, self.pixel_size_um_2d, target="nuclei")
            lab = np.asarray(labeled.cpu() if hasattr(labeled, "cpu") else labeled).squeeze()
            if lab.ndim == 3:  # (n_outputs, H, W) -> take the first output
                lab = lab[0]
            z_masks.append(lab.astype(np.uint32))
        stacked = np.stack(z_masks, axis=0)
        stitched = cellpose_utils.stitch3D(stacked, stitch_threshold=stitch_threshold)
        return self._save_and_record("instanseg_2dstitch", stitched)

    def micro_sam_3d(self, model_type="vit_b_lm"): # microSAM, true 3d, no custom model
        key = f"microsam_{model_type}"
        if key not in _MODEL_CACHE:
            _MODEL_CACHE[key] = get_predictor_and_segmenter(model_type=model_type)
        predictor, segmenter = _MODEL_CACHE[key]

        seg = automatic_3d_segmentation(self.dapi_ds.astype(np.float32), predictor, segmenter)
        return self._save_and_record("microsam_3d", seg)

    ######### MAIN PART############
    def run_all(self,include=("cpsam_2dstitch", "cpsam_true3d", "cellpose_custom_3d", "instanseg", "microsam")):
        self.rescale_image()
        runners = {
            "cpsam_2dstitch": self.cellpose_sam_2d_stitched,
            "cpsam_true3d": self.cellpose_sam_true_3d,
            "cellpose_custom_3d": self.cellpose_custom_3d,
            "instanseg": self.instanseg_2d_stitched,
            "microsam": self.micro_sam_3d,
        }
        for name in include:
            try:
                t0 = time.time()
                runners[name]()
                print(f"  {name} took {time.time() - t0:.1f}s")
                time_taken = time.time() - t0
                results = {"method": name, "time_taken": time_taken, "objects_found": self.counts}
            except Exception as exc:
                print(f"[SKIP] {name}: {type(exc).__name__}: {exc}")

        print("\ncounts:", self.counts)
        return self.counts

In [24]:
input_csv = pd.read_excel(r"z:\Bel\Jorge_SPACEFISH_Examples\image_locations.xlsx")
input_csv.head()


,image_name,path,dapi_channel,bf_channel,image_type,scene_id,"censor region (z1,z2,x1_z1, x2_z1, y1_z1,y2_z1,x1_z2,x2_z2,y1_z2,y2_z2)",existing_segmentation_path,2_channel_tif_save_path
0,dev4_1_6h,Z:\Jorge\20241124_6h_dev4\20241124_dev4_6h_mer...,6,7,vascu,0,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
1,dev4_2_6h,Z:\Jorge\20241125_repeats_6h_2d_pin255\2024112...,6,7,vascu,0,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
2,dev4_3_6h,Z:\Jorge\20241124_6h_dev4\20241124_dev4_6h_mer...,6,7,vascu,2,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
3,dev7_3_day1,Z:\Jorge\20241126_day1_dev7\20241126_dev7_day1...,6,7,vascu,0,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
4,dev7_2_day1,Z:\Jorge\20241126_day1_dev7\20241126_dev7_day1...,6,7,vascu,1,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...


In [ ]:
comparison = SegmentationComparisons(input_csv, index=0, scale_factor_xy=3, scale_factor_z=2)

# run everything (each method is wrapped in try/except, so one failure won't stop the rest)
comparison.run_all()

# or run individual methods:
# comparison.rescale_image()
# comparison.cellpose_sam_2d_stitched()
# comparison.cellpose_sam_true_3d()
# comparison.cellpose_custom_3d()
# comparison.instanseg_2d_stitched()
# comparison.micro_sam_3d()